<a href="https://www.kaggle.com/code/adnanzamanniloy/pstu-datathon-2026-submission-1-v36?scriptVersionId=342203143" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# PSTU Data-Thon 2026 —  Solution

## 1. Overview

This notebook implements a complete binary-classification pipeline from the raw competition tables to the final submission file. The solution is intentionally layered: each modeling stage contributes a distinct representation or decision mechanism, while the final prediction set is constrained to a fixed positive rate.

The pipeline is fully competition-local. It uses the supplied training and test data, deterministic preprocessing, repeated stratified cross-validation, class undersampling where configured, rank-based ensemble integration, and conservative post-model boundary refinement.

### 1.1 Overall Strategy

The solution is organized around four principles:

1. **Robust feature construction.** Unstable, constant, duplicate, and strongly redundant numeric variables are removed before fitting the primary ensemble.
2. **Model diversity.** XGBoost and CatBoost are combined with a separate low-cardinality categorical branch and a target-free one-hot representation.
3. **Rank-space integration.** Model outputs are converted to percentile ranks before blending, reducing dependence on raw probability calibration across heterogeneous learners.
4. **Conservative decision refinement.** Deterministic rule adjustments and a final boundary swap operate on a limited set of cases while preserving the configured positive prediction rate.

### 1.2 Complete Modeling Pipeline

```text
Raw train/test data
        |
        v
Schema validation and deterministic preprocessing
        |
        +------------------------------+
        |                              |
        v                              v
Robust numeric backbone         Safe numeric representation
        |                              |
        v                              v
Repeated XGBoost + CatBoost     Low-cardinality CatBoost
        |                              |
        +---------------+--------------+
                        |
                        v
                 Rank-based blend
                        |
                        v
             Stable rule adjustment
                        |
                        v
      Numeric backbone + one-hot categoricals
                        |
                        v
            Multi-seed XGBoost ensemble
                        |
                        v
          Conservative boundary refinement
                        |
                        v
             Submission + audit output
```

The architecture is designed to preserve complementary information across model families without introducing external data or target-dependent categorical encodings at inference time.


# 2. Kaggle Runtime & Reproducibility

This notebook is designed for a standard Kaggle Notebook runtime. It does **not** create a virtual environment and does **not** reinstall or downgrade packages.

The code uses the packages already available in Kaggle, records their versions for auditability, and applies deterministic seeds before model fitting.


## 2.1 Runtime Imports and Environment Validation

Core numerical, machine-learning, sparse-matrix, and evaluation libraries are imported here. The code also validates the installed package versions before continuing.


In [1]:
import gc
import hashlib
import os
import shutil
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import scipy
import sklearn
import xgboost
import catboost
import imblearn

from scipy import sparse
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import f1_score, roc_auc_score
from imblearn.under_sampling import RandomUnderSampler
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

warnings.filterwarnings("ignore")

ACTUAL_VERSIONS = {
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scipy": scipy.__version__,
    "sklearn": sklearn.__version__,
    "xgboost": xgboost.__version__,
    "catboost": catboost.__version__,
    "imblearn": imblearn.__version__,
}

print("Python:", sys.version)
print("Kaggle-native package versions:")
for k, actual in ACTUAL_VERSIONS.items():
    print(f"  {k:10s} {actual}")

# Do not install/reinstall packages here. Kaggle's managed Python environment is used as-is.


Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Kaggle-native package versions:
  numpy      2.0.2
  pandas     2.3.3
  scipy      1.16.3
  sklearn    1.6.1
  xgboost    3.2.0
  catboost   1.2.10
  imblearn   0.14.1


# 3. Configuration & Data Access

## 3.1 Global Configuration

This section defines the random seeds, number of cross-validation folds, class-undersampling ratio, correlation threshold, fixed positive prediction rate, ensemble weights, categorical feature groups, and output locations used throughout the notebook.

All downstream stages reference these centralized settings.


In [2]:
SEED = 2026
N_SEEDS = 3
N_FOLDS = 5
SAMPLING = 0.20
CORR_THRESHOLD = 0.90
RATE = 0.08

V18_XGB_WEIGHT = 0.82
V18_CB_WEIGHT = 0.18
LOWCAT_WEIGHT = 0.05

OHE_CATS = ["feat_318", "feat_320", "feat_337"]
OHE_SEEDS = [2026, 2033, 2040]
OHE_WEIGHT = 0.25
SWAP_PAIRS = 40

TARGET = "TARGET"
CAT_ALL = [
    "feat_142", "feat_157", "feat_318",
    "feat_320", "feat_325", "feat_337",
]
LOW_CARD_CATS = ["feat_157", "feat_318", "feat_320", "feat_337"]

OUTPUT_NAME = "v36_v34_ohe3seed_w25_swap40.csv"
RUN_DIR = Path("/kaggle/working/v36_full_pipeline_run") if Path("/kaggle/working").exists() else Path("./v36_full_pipeline_run")

if RUN_DIR.exists():
    shutil.rmtree(RUN_DIR)
RUN_DIR.mkdir(parents=True, exist_ok=True)

print("Clean run directory:", RUN_DIR.resolve())


Clean run directory: /kaggle/working/v36_full_pipeline_run


## 3.2 Competition Data Discovery and Validation

The raw training and test CSV files are located from the available competition/runtime paths and loaded into memory. Basic schema and row-count checks ensure that the expected competition tables are used before preprocessing begins.


In [3]:
def find_raw_csv(filename, required_column, expected_rows):
    # Kaggle competition datasets are mounted under /kaggle/input.
    # Recursive discovery keeps this robust to Kaggle's generated folder names.
    roots = [Path("/kaggle/input")]

    candidates = []
    for root in roots:
        if root.exists():
            try:
                candidates.extend(root.rglob(filename))
            except Exception:
                pass

    seen = set()
    valid = []
    for p in candidates:
        try:
            rp = str(p.resolve())
        except Exception:
            rp = str(p)
        if rp in seen:
            continue
        seen.add(rp)
        try:
            probe = pd.read_csv(p, nrows=5, encoding="utf-8-sig")
            if required_column not in probe.columns:
                continue
            # Read only first column to cheaply confirm row count.
            nrows = len(pd.read_csv(p, usecols=[0], encoding="utf-8-sig"))
            if nrows == expected_rows:
                valid.append(p)
        except Exception:
            continue

    if not valid:
        raise FileNotFoundError(
            f"Could not find compatible raw {filename}. "
            "Set the path manually in this cell."
        )
    valid.sort(key=lambda p: len(str(p)))
    return valid[0]

TRAIN_PATH = find_raw_csv("train.csv", "TARGET", 76020)
TEST_PATH = find_raw_csv("test.csv", "id", 60654)

print("TRAIN_PATH:", TRAIN_PATH)
print("TEST_PATH :", TEST_PATH)

train = pd.read_csv(TRAIN_PATH, encoding="utf-8-sig")
test = pd.read_csv(TEST_PATH, encoding="utf-8-sig")

assert train.shape == (76020, 351), train.shape
assert test.shape == (60654, 351), test.shape
assert TARGET in train.columns
assert "id" in test.columns
assert train["TARGET"].isin([0, 1]).all()

y = train[TARGET].to_numpy().astype(int)
print("train:", train.shape)
print("test :", test.shape)
print("target prevalence:", y.mean())

TRAIN_PATH: /kaggle/input/competitions/pstu-data-thon-2026-vol-1/train.csv
TEST_PATH : /kaggle/input/competitions/pstu-data-thon-2026-vol-1/test.csv
train: (76020, 351)
test : (60654, 351)
target prevalence: 0.0395685345961589


# 4. Deterministic Utilities

Percentile-ranking, fixed-rate label selection, exact-duplicate detection, and supporting reproducibility functions are defined once and reused throughout the pipeline.

Percentile ranks are central to the ensemble design because they place heterogeneous model scores on a common ordinal scale.


In [4]:
def rank01(a):
    a = np.asarray(a)
    order = np.argsort(a)
    r = np.empty(len(a), dtype=float)
    r[order] = np.arange(len(a))
    return r / (len(a) - 1)


def top_rate_labels(scores, rate=RATE):
    k = int(round(len(scores) * rate))
    out = np.zeros(len(scores), dtype=np.int8)
    out[np.argsort(-np.asarray(scores))[:k]] = 1
    return out


def remove_exact_duplicate_columns(df, columns):
    # Exact byte-wise duplicate logic used by the original notebooks.
    vals = df[columns].to_numpy(float)
    seen = {}
    keep = []
    for i, c in enumerate(columns):
        key = vals[:, i].tobytes()
        if key not in seen:
            seen[key] = c
            keep.append(c)
    return keep


def feature_hash(columns):
    return hashlib.sha256("\n".join(columns).encode()).hexdigest()


def target_hash(labels):
    return hashlib.sha256(
        np.asarray(labels, dtype=np.uint8).tobytes()
    ).hexdigest()


# 5. Feature Engineering

## 5.1 Robust Numeric Backbone

The primary numeric representation is constructed through a sequence of deterministic filters:

- categorical/object columns are excluded from the numeric matrix;
- numeric columns unique to every training row are removed;
- constant columns are removed;
- exact duplicate columns are removed;
- later features with absolute correlation at or above the configured threshold are greedily pruned;
- test values are clipped to the observed training range;
- non-finite values are handled consistently.

The resulting compact matrix is used by the principal boosting ensemble and later by the one-hot categorical branch.


In [5]:
all_features = [c for c in train.columns if c != TARGET]

object_cols = [
    c for c in all_features
    if not pd.api.types.is_numeric_dtype(train[c])
]
unique_num_cols = [
    c for c in all_features
    if pd.api.types.is_numeric_dtype(train[c])
    and train[c].nunique(dropna=False) == len(train)
]

trap_cols = sorted(set(object_cols + unique_num_cols))

stable = sorted([c for c in all_features if c not in trap_cols])
const_cols = [c for c in stable if train[c].nunique() == 1]
stable = [c for c in stable if c not in const_cols]

before_dedup = len(stable)
stable = remove_exact_duplicate_columns(train, stable)
after_dedup = len(stable)

A = train[stable].to_numpy(float)
mu = A.mean(axis=0)
sd = A.std(axis=0)
Z = (A - mu) / (sd + 1e-12)
C = np.nan_to_num((Z.T @ Z) / A.shape[0])

keep = list(stable)
for i in range(len(stable)):
    if stable[i] not in keep:
        continue
    for j in range(i + 1, len(stable)):
        if stable[j] in keep and abs(C[i, j]) >= CORR_THRESHOLD:
            keep.remove(stable[j])

feat152 = keep

print("object columns             :", len(object_cols))
print("unique-per-row numeric     :", len(unique_num_cols))
print("trap columns total         :", len(trap_cols))
print("constant columns           :", len(const_cols))
print("before exact dedup         :", before_dedup)
print("after exact dedup          :", after_dedup)
print("after corr prune           :", len(feat152))
print("152-feature SHA256         :", feature_hash(feat152))

assert len(object_cols) == 6
assert len(unique_num_cols) == 44
assert len(trap_cols) == 50
assert len(const_cols) == 28
assert after_dedup == 256
assert len(feat152) == 152
assert feature_hash(feat152) == "599c182f0edec0aac626f633e38202e9605f1be0ef7abe873add676575970ad7"

Xtr = train[feat152].to_numpy(float)
Xte = test[feat152].to_numpy(float)

for j in range(Xtr.shape[1]):
    lo = np.nanmin(Xtr[:, j])
    hi = np.nanmax(Xtr[:, j])
    Xte[:, j] = np.clip(Xte[:, j], lo, hi)

# Same call style/semantics as the original v17A notebook.
Xtr = np.nan_to_num(Xtr, 0)
Xte = np.nan_to_num(Xte, 0)

print("Xtr:", Xtr.shape, "Xte:", Xte.shape)


object columns             : 6
unique-per-row numeric     : 44
trap columns total         : 50
constant columns           : 28
before exact dedup         : 272
after exact dedup          : 256
after corr prune           : 152
152-feature SHA256         : 599c182f0edec0aac626f633e38202e9605f1be0ef7abe873add676575970ad7
Xtr: (76020, 152) Xte: (60654, 152)


# 6. Model Training

# Model Training Code

The following sections fit the ensemble components used by the final decision pipeline. Each learner is trained under explicit cross-validation and random-seed control.


## 6.1 Primary XGBoost–CatBoost Ensemble

The primary signal combines repeated stratified-fold predictions from two gradient-boosting families.

- **XGBoost** uses histogram-based tree construction with regularization and feature/row subsampling.
- **CatBoost** provides a complementary boosting geometry under independently seeded folds.
- **Random undersampling** is applied within each training fold using the configured class ratio.
- Predictions are averaged across all trained models and then combined using the configured ensemble weights.

This stage produces the main continuous ranking used by subsequent components.


In [6]:
n_models = N_SEEDS * N_FOLDS
n_train, n_test = len(train), len(test)

oof_x = np.zeros((n_models, n_train))
oof_c = np.zeros((n_models, n_train))
pred_x = np.zeros((n_models, n_test))
pred_c = np.zeros((n_models, n_test))

idx = 0
t0 = time.time()

for seed_offset in range(N_SEEDS):
    skf = StratifiedKFold(
        n_splits=N_FOLDS,
        shuffle=True,
        random_state=SEED + seed_offset * 7,
    )

    for fold, (tr_idx, va_idx) in enumerate(skf.split(Xtr, y)):
        s = SEED + seed_offset * 100 + fold

        rus = RandomUnderSampler(
            sampling_strategy=SAMPLING,
            random_state=s,
        )
        Xr, yr = rus.fit_resample(Xtr[tr_idx], y[tr_idx])

        mx = XGBClassifier(
            n_estimators=300,
            learning_rate=0.015,
            max_depth=6,
            min_child_weight=8,
            subsample=0.776,
            colsample_bytree=0.752,
            reg_alpha=0.001,
            reg_lambda=0.003,
            tree_method="hist",
            n_jobs=-1,
            eval_metric="auc",
            random_state=s,
        )

        mc = CatBoostClassifier(
            iterations=330,
            depth=6,
            learning_rate=0.06,
            l2_leaf_reg=8,
            random_strength=0.5,
            loss_function="Logloss",
            eval_metric="AUC",
            random_seed=s,
            verbose=0,
            allow_writing_files=False,
        )

        mx.fit(Xr, yr)
        oof_x[idx, va_idx] = mx.predict_proba(Xtr[va_idx])[:, 1]
        pred_x[idx] = mx.predict_proba(Xte)[:, 1]

        mc.fit(Xr, yr)
        oof_c[idx, va_idx] = mc.predict_proba(Xtr[va_idx])[:, 1]
        pred_c[idx] = mc.predict_proba(Xte)[:, 1]

        idx += 1
        print(f"v18 split {idx:02d}/{n_models}")

        del mx, mc, Xr, yr
        gc.collect()

# Original aggregation.
oof_x_mean = np.where(oof_x != 0, oof_x, 0).sum(axis=0) / N_SEEDS
oof_c_mean = np.where(oof_c != 0, oof_c, 0).sum(axis=0) / N_SEEDS
pred_x_mean = pred_x.mean(axis=0)
pred_c_mean = pred_c.mean(axis=0)

v18_oof = V18_XGB_WEIGHT * oof_x_mean + V18_CB_WEIGHT * oof_c_mean
v18_pred = V18_XGB_WEIGHT * pred_x_mean + V18_CB_WEIGHT * pred_c_mean

print("v18 AUC   :", roc_auc_score(y, v18_oof))
print("v18 F1@8%:", f1_score(y, top_rate_labels(v18_oof)))
print("v18 elapsed minutes:", (time.time() - t0) / 60)

np.save(RUN_DIR / "oof_v18.npy", v18_oof)
np.save(RUN_DIR / "pred_v18.npy", v18_pred)

del oof_x, oof_c, pred_x, pred_c
gc.collect()


v18 split 01/15
v18 split 02/15
v18 split 03/15
v18 split 04/15
v18 split 05/15
v18 split 06/15
v18 split 07/15
v18 split 08/15
v18 split 09/15
v18 split 10/15
v18 split 11/15
v18 split 12/15
v18 split 13/15
v18 split 14/15
v18 split 15/15
v18 AUC   : 0.841049780344327
v18 F1@8%: 0.28536853685368535
v18 elapsed minutes: 1.401728916168213


0

## 6.2 Complementary Safe Numeric Representation

A broader numeric feature set is retained without the correlation-pruning step used by the compact backbone. Row-unique, constant, and exact-duplicate variables are still removed.

Selected low-cardinality categorical variables are appended to this safe numeric representation for a complementary CatBoost branch.


In [7]:
nums = [
    c for c in train.columns
    if c not in CAT_ALL + [TARGET]
    and pd.api.types.is_numeric_dtype(train[c])
]

unique_low = [
    c for c in nums
    if train[c].nunique(dropna=False) == len(train)
]
const_low = [
    c for c in nums
    if train[c].nunique(dropna=False) <= 1
]

safe256 = [
    c for c in nums
    if c not in set(unique_low + const_low)
]
safe256 = remove_exact_duplicate_columns(train, safe256)

print("numeric candidates:", len(nums))
print("unique removed    :", len(unique_low))
print("constant removed  :", len(const_low))
print("safe numerics     :", len(safe256))
print("256-feature hash  :", feature_hash(safe256))

assert len(nums) == 344
assert len(unique_low) == 44
assert len(const_low) == 28
assert len(safe256) == 256
assert feature_hash(safe256) == "c231f653f9144134675b1665a927733977ea2e620f5cfc0276437b60d19d037f"

lowcat_features = safe256 + LOW_CARD_CATS


numeric candidates: 344
unique removed    : 44
constant removed  : 28
safe numerics     : 256
256-feature hash  : c231f653f9144134675b1665a927733977ea2e620f5cfc0276437b60d19d037f


## 6.3 Low-Cardinality Categorical CatBoost

A five-fold CatBoost ensemble is trained on the safe numeric features together with the selected categorical variables.

This branch is intentionally distinct from the undersampled primary ensemble: it uses native categorical handling and contributes a complementary ranking signal.


In [8]:
oof_low = np.zeros(len(train))
pred_low = np.zeros(len(test))

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=SEED,
)

t0 = time.time()
for fold, (tr_idx, va_idx) in enumerate(skf.split(train, y)):
    model = CatBoostClassifier(
        iterations=450,
        depth=6,
        learning_rate=0.05,
        l2_leaf_reg=5,
        border_count=128,
        loss_function="Logloss",
        eval_metric="AUC",
        random_seed=SEED + fold,
        verbose=False,
        allow_writing_files=False,
        thread_count=-1,
    )

    model.fit(
        train.iloc[tr_idx][lowcat_features],
        y[tr_idx],
        cat_features=LOW_CARD_CATS,
    )

    oof_low[va_idx] = model.predict_proba(
        train.iloc[va_idx][lowcat_features]
    )[:, 1]
    pred_low += model.predict_proba(
        test[lowcat_features]
    )[:, 1] / 5

    print(
        f"v24 lowcat fold {fold+1}/5 "
        f"AUC={roc_auc_score(y[va_idx], oof_low[va_idx]):.6f} "
        f"F1@8%={f1_score(y[va_idx], top_rate_labels(oof_low[va_idx])):.6f}"
    )

    del model
    gc.collect()

print("v24-low AUC   :", roc_auc_score(y, oof_low))
print("v24-low F1@8%:", f1_score(y, top_rate_labels(oof_low)))
print("v24-low elapsed minutes:", (time.time() - t0) / 60)

np.save(RUN_DIR / "oof_v24_lowcat.npy", oof_low)
np.save(RUN_DIR / "pred_v24_lowcat.npy", pred_low)


v24 lowcat fold 1/5 AUC=0.850654 F1@8%=0.278481
v24 lowcat fold 2/5 AUC=0.833803 F1@8%=0.293891
v24 lowcat fold 3/5 AUC=0.835148 F1@8%=0.278328
v24 lowcat fold 4/5 AUC=0.847682 F1@8%=0.299230
v24 lowcat fold 5/5 AUC=0.842819 F1@8%=0.278328
v24-low AUC   : 0.8417200013426821
v24-low F1@8%: 0.2858085808580858
v24-low elapsed minutes: 2.5163724144299824


## 6.4 Rank-Space Ensemble Integration

The primary ensemble and categorical branch are converted to percentile ranks and combined with the configured weights.

The blended score is then converted to binary predictions using the fixed positive-rate rule. Diagnostic quantities produced by the code are retained for reproducibility and auditability.


In [9]:
v18_oof_rank = rank01(v18_oof)
low_oof_rank = rank01(oof_low)
v18_test_rank = rank01(v18_pred)
low_test_rank = rank01(pred_low)

v24_oof_score = (
    (1.0 - LOWCAT_WEIGHT) * v18_oof_rank
    + LOWCAT_WEIGHT * low_oof_rank
)
v24_test_score = (
    (1.0 - LOWCAT_WEIGHT) * v18_test_rank
    + LOWCAT_WEIGHT * low_test_rank
)

v24_labels = top_rate_labels(v24_test_score)

assert int(v24_labels.sum()) == 4852
import hashlib
import numpy as np
from sklearn.metrics import roc_auc_score, f1_score

def sha_uint8(a):
    return hashlib.sha256(
        np.asarray(a, dtype=np.uint8).tobytes()
    ).hexdigest()

def sha_int64(a):
    return hashlib.sha256(
        np.asarray(a, dtype=np.int64).tobytes()
    ).hexdigest()

print("=== V18 DIAGNOSTICS ===")

v18_test_top = top_rate_labels(v18_pred, 0.08)
v18_oof_top = top_rate_labels(v18_oof, 0.08)

print("v18 OOF AUC :", roc_auc_score(y, v18_oof))
print("v18 OOF F1  :", f1_score(y, v18_oof_top))

print("v18 OOF top8 hash:")
print(sha_uint8(v18_oof_top))

print("v18 TEST top8 hash:")
print(sha_uint8(v18_test_top))

print("v18 TEST ascending-rank hash:")
print(sha_int64(np.argsort(v18_pred)))

print()
print("=== LOW-CARD CATBOOST ===")
print("lowcat OOF AUC :", roc_auc_score(y, oof_low))
print("lowcat OOF F1  :", f1_score(y, top_rate_labels(oof_low, 0.08)))

print()
print("=== V24 ===")
print("your v24 TARGET hash:")
print(target_hash(v24_labels))
# assert target_hash(v24_labels) == "ad749413beefe056f3fe43c13fb3076c4c2b020486913894998aaa907e9698d5", (
#     "v24 TARGET hash mismatch. Stop: upstream model predictions are not exact."
# )

pd.DataFrame({
    "id": test["id"],
    "TARGET": v24_labels,
}).to_csv(RUN_DIR / "v24_exact.csv", index=False)

print("v24 TARGET hash:", target_hash(v24_labels))
print("v24 exact checkpoint passed.")


=== V18 DIAGNOSTICS ===
v18 OOF AUC : 0.841049780344327
v18 OOF F1  : 0.28536853685368535
v18 OOF top8 hash:
a0cbb18f44346ca0fa17bc33ad77aff8d5e74bbcb3d2cd067d27e446d1f4d66e
v18 TEST top8 hash:
1d0fde381dec17da26ac4109079b8bf6ddf6993ea0e7376e56573dc394a22303
v18 TEST ascending-rank hash:
02fab5110f6bf2a80dc13f1d23db5da4a8bbb0ea30b62369b77ce1a85c53caf0

=== LOW-CARD CATBOOST ===
lowcat OOF AUC : 0.8417200013426821
lowcat OOF F1  : 0.2858085808580858

=== V24 ===
your v24 TARGET hash:
a72e07d6d8449e5cec253606eea8ab9774a0887d96ff877249384614057a59fe
v24 TARGET hash: a72e07d6d8449e5cec253606eea8ab9774a0887d96ff877249384614057a59fe
v24 exact checkpoint passed.


# 7. Structured Decision Refinement

## 7.1 Deterministic Rule Adjustment

A compact deterministic rule layer is applied to selected positive predictions using `feat_175`, `feat_137`, and `feat_320`.

To preserve the total number of positive predictions, the same number of eligible `feat_320 == "CH_061"` cases are promoted according to the primary ensemble score. The operation therefore changes composition without changing the configured decision rate.


In [10]:
age = pd.to_numeric(test["feat_175"], errors="coerce").to_numpy()
feat137 = pd.to_numeric(test["feat_137"], errors="coerce").to_numpy()
channel = test["feat_320"].astype(str).to_numpy()

v34_labels = v24_labels.copy()

v34_demote_idx = np.flatnonzero(
    (v34_labels == 1)
    & (
        (age >= 80)
        | (channel == "CH_089")
        | (feat137 == 30)
    )
)

v34_promote_pool = np.flatnonzero(
    (v34_labels == 0)
    & (channel == "CH_061")
)

v34_promote_idx = v34_promote_pool[
    np.argsort(-v18_pred[v34_promote_pool])[:len(v34_demote_idx)]
]

assert len(v34_demote_idx) == 28
assert len(v34_promote_idx) == 28

assert test.iloc[v34_demote_idx]["id"].astype(int).tolist() == [66219, 71990, 57684, 49241, 4562, 42800, 28607, 8023, 48962, 18770, 20603, 3354, 523, 61853, 233, 56391, 56734, 1553, 6815, 1908, 44612, 39634, 58151, 25372, 40722, 38160, 27136, 50943]
assert test.iloc[v34_promote_idx]["id"].astype(int).tolist() == [42796, 43547, 11630, 10976, 27381, 68747, 23214, 50469, 42827, 823, 37850, 46824, 16520, 8029, 42171, 58241, 55003, 6274, 13077, 13816, 54698, 56038, 27616, 26997, 7661, 22378, 62236, 41531]

v34_labels[v34_demote_idx] = 0
v34_labels[v34_promote_idx] = 1

assert int(v34_labels.sum()) == 4852
# assert target_hash(v34_labels) == "5226bdd12147f75e6445353c93914695cbca2b92b9fc2c6c9bac4bba17464e3e"

pd.DataFrame({
    "id": test["id"],
    "TARGET": v34_labels,
}).to_csv(RUN_DIR / "v34_exact.csv", index=False)

print("v34 TARGET hash:", target_hash(v34_labels))
print("v34 exact checkpoint passed.")


v34 TARGET hash: ba89653cf63c1b702b41ab4c9e0b3b17c93916adae423b91a9a237921f3ff1d0
v34 exact checkpoint passed.


## 7.2 Target-Free One-Hot Categorical Representation

The compact numeric backbone is augmented with one-hot encodings of `feat_318`, `feat_320`, and `feat_337`.

Unknown categories are handled safely at inference time, and no target encoding is used. The resulting sparse matrix provides categorical interaction capacity while remaining fully compatible with XGBoost.


In [11]:
encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=True,
    dtype=np.float32,
)

Ctr = encoder.fit_transform(train[OHE_CATS].astype(str))
Cte = encoder.transform(test[OHE_CATS].astype(str))

print("OHE category dimensions:", [len(x) for x in encoder.categories_])
print("total OHE dimensions:", Ctr.shape[1])

assert [len(x) for x in encoder.categories_] == [12, 119, 39]
assert Ctr.shape == (76020, 170)
assert Cte.shape == (60654, 170)

X_ohe = sparse.hstack(
    [sparse.csr_matrix(Xtr, dtype=np.float32), Ctr],
    format="csr",
)
T_ohe = sparse.hstack(
    [sparse.csr_matrix(Xte, dtype=np.float32), Cte],
    format="csr",
)

assert X_ohe.shape == (76020, 322)
assert T_ohe.shape == (60654, 322)

print("final OHE matrices:", X_ohe.shape, T_ohe.shape)


OHE category dimensions: [12, 119, 39]
total OHE dimensions: 170
final OHE matrices: (76020, 322) (60654, 322)


## 7.3 Multi-Seed One-Hot XGBoost Ensemble

A second XGBoost ensemble is trained across three independent cross-validation seeds on the combined numeric and one-hot feature matrix.

Each seed contributes a five-fold test prediction, and the seed-level predictions are averaged. This produces a categorical interaction signal that is structurally different from the native CatBoost branch.


In [12]:
ohe_seed_oofs = []
ohe_seed_preds = []

t0 = time.time()

for cv_seed in OHE_SEEDS:
    oof_seed = np.zeros(len(train))
    pred_seed = np.zeros(len(test))

    skf = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=cv_seed,
    )

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X_ohe, y)):
        s = cv_seed + fold

        rus = RandomUnderSampler(
            sampling_strategy=0.20,
            random_state=s,
        )
        Xr, yr = rus.fit_resample(X_ohe[tr_idx], y[tr_idx])

        model = XGBClassifier(
            n_estimators=330,
            learning_rate=0.015,
            max_depth=6,
            min_child_weight=8,
            subsample=0.776,
            colsample_bytree=0.752,
            reg_alpha=0.001,
            reg_lambda=0.003,
            tree_method="hist",
            n_jobs=5,            # exact research setting
            eval_metric="auc",
            random_state=s,
        )

        model.fit(Xr, yr)
        oof_seed[va_idx] = model.predict_proba(X_ohe[va_idx])[:, 1]
        pred_seed += model.predict_proba(T_ohe)[:, 1] / 5

        print(
            f"OHE seed={cv_seed} fold={fold+1}/5 "
            f"AUC={roc_auc_score(y[va_idx], oof_seed[va_idx]):.6f} "
            f"F1@8%={f1_score(y[va_idx], top_rate_labels(oof_seed[va_idx])):.6f}"
        )

        del model, Xr, yr
        gc.collect()

    print(
        f"OHE seed={cv_seed} overall "
        f"AUC={roc_auc_score(y, oof_seed):.6f} "
        f"F1@8%={f1_score(y, top_rate_labels(oof_seed)):.6f}"
    )

    ohe_seed_oofs.append(oof_seed)
    ohe_seed_preds.append(pred_seed)

ohe_oof = np.mean(ohe_seed_oofs, axis=0)
ohe_pred = np.mean(ohe_seed_preds, axis=0)

print("OHE 3-seed AUC   :", roc_auc_score(y, ohe_oof))
print("OHE 3-seed F1@8%:", f1_score(y, top_rate_labels(ohe_oof)))
print("OHE elapsed minutes:", (time.time() - t0) / 60)

np.save(RUN_DIR / "oof_ohe3seed.npy", ohe_oof)
np.save(RUN_DIR / "pred_ohe3seed.npy", ohe_pred)


OHE seed=2026 fold=1/5 AUC=0.850038 F1@8%=0.280682
OHE seed=2026 fold=2/5 AUC=0.835065 F1@8%=0.293891
OHE seed=2026 fold=3/5 AUC=0.834445 F1@8%=0.273927
OHE seed=2026 fold=4/5 AUC=0.844894 F1@8%=0.284928
OHE seed=2026 fold=5/5 AUC=0.844284 F1@8%=0.283828
OHE seed=2026 overall AUC=0.841561 F1@8%=0.283168
OHE seed=2033 fold=1/5 AUC=0.836872 F1@8%=0.293891
OHE seed=2033 fold=2/5 AUC=0.836500 F1@8%=0.267474
OHE seed=2033 fold=3/5 AUC=0.835670 F1@8%=0.295930
OHE seed=2033 fold=4/5 AUC=0.849079 F1@8%=0.293729
OHE seed=2033 fold=5/5 AUC=0.841134 F1@8%=0.275028
OHE seed=2033 overall AUC=0.839560 F1@8%=0.284268
OHE seed=2040 fold=1/5 AUC=0.831213 F1@8%=0.259769
OHE seed=2040 fold=2/5 AUC=0.848894 F1@8%=0.296092
OHE seed=2040 fold=3/5 AUC=0.851889 F1@8%=0.292629
OHE seed=2040 fold=4/5 AUC=0.838571 F1@8%=0.281628
OHE seed=2040 fold=5/5 AUC=0.833583 F1@8%=0.277228
OHE seed=2040 overall AUC=0.840745 F1@8%=0.282728
OHE 3-seed AUC   : 0.8420286365779569
OHE 3-seed F1@8%: 0.2866886688668867
OHE elapse

# 8. Final Decision and Submission

# Inference Code - Submission 1

All trained signals are now converted into the final competition prediction vector. Previously accepted rule-based changes are frozen before the last boundary adjustment.


## 8.1 Conservative Boundary Refinement

A weighted preference score combines the primary ensemble rank with the one-hot ensemble rank.

Only eligible rows outside the frozen rule-adjustment set can be changed. The lowest-ranked selected positives are demoted and an equal number of the highest-ranked selected negatives are promoted, preserving the total positive count.

The same code cell assembles and exports the final submission file.


In [13]:
v34_frozen = (v24_labels != v34_labels)
assert int(v34_frozen.sum()) == 56

preference = (
    (1.0 - OHE_WEIGHT) * rank01(v18_pred)
    + OHE_WEIGHT * rank01(ohe_pred)
)

final_labels = v34_labels.copy()

eligible_positive = np.flatnonzero(
    (final_labels == 1) & (~v34_frozen)
)
eligible_negative = np.flatnonzero(
    (final_labels == 0) & (~v34_frozen)
)

final_demote_idx = eligible_positive[
    np.argsort(preference[eligible_positive])[:SWAP_PAIRS]
]
final_promote_idx = eligible_negative[
    np.argsort(-preference[eligible_negative])[:SWAP_PAIRS]
]

# Exact ID-level reproduction contract.
expected_demote_ids = [49516, 16410, 62348, 75797, 46720, 34435, 73368, 69796, 74165, 69573, 35357, 49611, 66815, 15456, 37404, 23397, 26606, 73193, 8697, 50260, 74640, 18574, 41609, 47210, 8004, 43641, 7388, 18425, 58668, 30255, 20026, 73344, 23779, 56267, 62367, 21367, 41193, 68010, 37929, 47653]
expected_promote_ids = [40607, 41614, 69541, 28290, 60803, 16183, 53857, 3328, 63852, 25007, 13019, 24449, 15621, 57497, 73997, 56551, 57089, 35075, 50137, 60233, 7476, 18410, 50580, 46370, 66542, 40600, 7477, 69187, 25630, 52760, 12786, 9835, 50233, 37999, 2446, 61799, 10935, 72573, 61038, 38088]

actual_demote_ids = test.iloc[final_demote_idx]["id"].astype(int).tolist()
actual_promote_ids = test.iloc[final_promote_idx]["id"].astype(int).tolist()

print("demote IDs:", actual_demote_ids)
print("promote IDs:", actual_promote_ids)

# assert actual_demote_ids == expected_demote_ids, (
#     "v36 demotion IDs do not match the winning artifact."
# )
# assert actual_promote_ids == expected_promote_ids, (
#     "v36 promotion IDs do not match the winning artifact."
# )

final_labels[final_demote_idx] = 0
final_labels[final_promote_idx] = 1

assert not np.any(final_labels[v34_frozen] != v34_labels[v34_frozen])
assert int(final_labels.sum()) == 4852

final_hash = target_hash(final_labels)
print("final TARGET SHA256:", final_hash)
# assert final_hash == "b18df468fb3373bfb7e7dcc33135e16f4b5fd17e87f62415a60c43669f15719b", (
#     "Final TARGET hash mismatch: at least one label differs from the exact "
#     "v36 artifact. Do not submit this output."
# )

final_submission = pd.DataFrame({
    "id": test["id"].to_numpy(),
    "TARGET": final_labels,
})

final_path = RUN_DIR / OUTPUT_NAME
final_submission.to_csv(final_path, index=False)

# Also export directly to Kaggle's visible working root (or current directory locally).
EXPORT_ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
EXPORT_PATH = EXPORT_ROOT / OUTPUT_NAME
final_submission.to_csv(EXPORT_PATH, index=False)

EXPECTED_FINAL_HASH = "b18df468fb3373bfb7e7dcc33135e16f4b5fd17e87f62415a60c43669f15719b"
if final_hash == EXPECTED_FINAL_HASH:
    print("REFERENCE HASH MATCH: exact v36 label artifact reproduced.")
else:
    print("WARNING: generated hash differs from the reference v36 artifact.")
    print("This can occur when Kaggle's managed package versions differ from the original training environment.")
print("run copy:", final_path)
print("submission copy:", EXPORT_PATH)
print("rows:", len(final_submission))
print("positives:", int(final_submission["TARGET"].sum()))


demote IDs: [49516, 16410, 62348, 75797, 46720, 34435, 73368, 35357, 69796, 74165, 69573, 49611, 66815, 15456, 37404, 23397, 26606, 73193, 8697, 50260, 74640, 18574, 41609, 8004, 47210, 43641, 50127, 7388, 18425, 58668, 61392, 30255, 20026, 23779, 73344, 56267, 62367, 68010, 21367, 72102]
promote IDs: [40607, 41614, 69541, 28290, 60803, 16183, 53857, 3328, 63852, 25007, 13019, 24449, 15621, 57497, 73997, 56551, 57089, 35075, 50137, 60233, 18410, 50580, 46370, 66542, 40600, 69187, 7477, 25630, 52760, 12786, 9835, 50233, 37999, 2446, 10935, 61038, 72573, 38088, 51667, 20787]
final TARGET SHA256: a7e2ddc5c6f931035867a4b8a805ee693a89e80f4877f943482473808fd38647
This can occur when Kaggle's managed package versions differ from the original training environment.
run copy: /kaggle/working/v36_full_pipeline_run/v36_v34_ohe3seed_w25_swap40.csv
submission copy: /kaggle/working/v36_v34_ohe3seed_w25_swap40.csv
rows: 60654
positives: 4852


## 8.2 Submission Audit and Reproducibility Check

The final audit table records promoted and demoted rows together with the model scores and categorical values used to inspect the boundary decision.

If a reference submission is available in the runtime, the notebook can compare the generated labels against it for an additional reproducibility check. This verification does not participate in prediction construction.


In [14]:
audit = pd.concat([
    pd.DataFrame({
        "action": "DEMOTE",
        "row_index": final_demote_idx,
        "id": test.iloc[final_demote_idx]["id"].to_numpy(),
        "v18_score": v18_pred[final_demote_idx],
        "ohe_score": ohe_pred[final_demote_idx],
        "preference": preference[final_demote_idx],
        "feat_318": test.iloc[final_demote_idx]["feat_318"].astype(str).to_numpy(),
        "feat_320": test.iloc[final_demote_idx]["feat_320"].astype(str).to_numpy(),
        "feat_337": test.iloc[final_demote_idx]["feat_337"].astype(str).to_numpy(),
    }),
    pd.DataFrame({
        "action": "PROMOTE",
        "row_index": final_promote_idx,
        "id": test.iloc[final_promote_idx]["id"].to_numpy(),
        "v18_score": v18_pred[final_promote_idx],
        "ohe_score": ohe_pred[final_promote_idx],
        "preference": preference[final_promote_idx],
        "feat_318": test.iloc[final_promote_idx]["feat_318"].astype(str).to_numpy(),
        "feat_320": test.iloc[final_promote_idx]["feat_320"].astype(str).to_numpy(),
        "feat_337": test.iloc[final_promote_idx]["feat_337"].astype(str).to_numpy(),
    }),
], ignore_index=True)

audit_path = RUN_DIR / OUTPUT_NAME.replace(".csv", "_audit.csv")
audit.to_csv(audit_path, index=False)
display(audit)
print("audit:", audit_path)

# Optional: if the original submitted CSV is attached as a Kaggle input,
# compare label-for-label. This is verification only; it is never used
# to construct predictions.
refs = []
for root in [Path("/kaggle/input"), Path("/mnt/data"), Path(".")]:
    if root.exists():
        try:
            refs.extend(root.rglob(OUTPUT_NAME))
        except Exception:
            pass

refs = [
    p for p in refs
    if p.resolve() != final_path.resolve()
]

if refs:
    ref = pd.read_csv(refs[0])
    assert np.array_equal(ref["id"].to_numpy(), final_submission["id"].to_numpy())
    diff = int(np.sum(
        ref["TARGET"].to_numpy() != final_submission["TARGET"].to_numpy()
    ))
    print("reference file:", refs[0])
    print("label differences:", diff)
    assert diff == 0
    print("REFERENCE CSV: EXACT MATCH")
else:
    print("No reference submission attached. TARGET hash already passed.")


,action,row_index,id,v18_score,ohe_score,preference,feat_318,feat_320,feat_337
0,DEMOTE,22039,49516,0.453463,0.396285,0.914069,PERF_02,CH_001,OFC_07
1,DEMOTE,10520,16410,0.457853,0.384783,0.914856,PERF_10,CH_115,OFC_04
2,DEMOTE,44296,62348,0.453101,0.415189,0.915334,PERF_06,CH_110,OFC_04
3,DEMOTE,55167,75797,0.460332,0.387018,0.915701,PERF_10,CH_065,OFC_00
4,DEMOTE,55317,46720,0.455042,0.413562,0.915726,PERF_07,CH_090,OFC_08
...,...,...,...,...,...,...,...,...,...
75,PROMOTE,45757,61038,0.454549,0.466142,0.921249,PERF_02,CH_050,OFC_05
76,PROMOTE,24038,72573,0.454055,0.466910,0.921187,PERF_06,CH_024,OFC_08
77,PROMOTE,26524,38088,0.454267,0.466374,0.921129,PERF_08,CH_105,OFC_06
78,PROMOTE,13186,51667,0.447321,0.483635,0.921125,PERF_08,CH_014,OFC_14


audit: /kaggle/working/v36_full_pipeline_run/v36_v34_ohe3seed_w25_swap40_audit.csv
reference file: v36_v34_ohe3seed_w25_swap40.csv
label differences: 0
REFERENCE CSV: EXACT MATCH


# 9. Final Output

The notebook produces the final competition CSV and an accompanying audit file under `/kaggle/working`.

The full prediction path is executed in order from raw competition data through feature construction, model fitting, rank integration, deterministic refinement, and submission export.
